[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Why NumPy


## What you will be able to do

Turn a year of weather readings into a NumPy array, then convert, average and search all 35,040
values without writing a loop, and see from your own run why the array version is the one to write.

## The idea

### The problem

A weather network logs a temperature every hour at four stations. A year of that is 8,760 readings
for each station, 35,040 numbers in all, and the questions you want answered are ordinary ones. What
is every reading in Fahrenheit? What was each station's mean for the year? When was the coldest hour?

With Python lists, each of those questions is a loop. Converting needs a loop inside a loop, one over
the stations and one over the hours, building a new list as it goes. The mean needs a running total
and a count, and a check for the hours a station sent nothing. The coldest hour needs a loop that
remembers the lowest value so far and where it was. None of these is hard, and each one takes several
lines that say how to walk through the data instead of what you want from it.

The loops are also slow. Python runs every step of a loop as a separate operation on a separate number
object, and 35,040 steps add up. A notebook that asks a dozen questions of a few years of readings
starts to wait on every cell.

### What a NumPy array is

> A **NumPy array** is a grid of values of one type, stored next to each other in a single block of
> memory, with a **shape** that gives how many values lie along each dimension. An operation on an
> array applies to every value in it, in a loop that NumPy runs in compiled code.

### Why it works that way

Because every value in an array has the same type, NumPy can store them as plain machine numbers,
eight bytes each for the decimals in this notebook, instead of as separate Python objects. Because
they sit next to each other, a compiled loop can walk through them without asking Python anything on
the way. That is what NumPy's documentation means by **vectorized**. The loop is still there, and
NumPy runs it for you.

The readings become one array of shape (4, 8760), four rows of stations by 8,760 columns of hours.
Converting all of them to Fahrenheit is then `readings * 9 / 5 + 32`, the formula as you would write
it on paper. The mean, the lowest value and the position of the lowest value are methods and
functions that take the whole array.

People who know lists are often surprised that the same operators do different things here. `+`
joins two lists end to end and adds two arrays value by value, and `*` with a list repeats it. An
array also holds one type for every value, so a missing reading cannot be `None`. NumPy uses `nan`, a floating
point value that means "not a number", and gives you functions such as `np.nanmean` that skip it.

### Where this shows up

Almost everywhere numbers are processed in Python. A column of a pandas table is a NumPy array
underneath, which the **Pandas, Deep Dive** guide builds on. Matplotlib plots arrays, scikit-learn
fits models to them, and an image loaded from a file is an array of pixel values. The readings in
this guide are the ones the **sqlite3, Deep Dive** guide stores in a database, made by the same
formula, so its numbers and this guide's agree hour by hour.

### What this notebook covers

- the readings as Python lists, and the same readings as one array
- every reading converted at once, and checked against the loops
- each station's mean, and what the missing hours do to it
- the coldest hour, found with one function
- daily means in one line, a preview of the **Shape** and **The axis Argument** notebooks
- the loops and the array timed against each other

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np

celsius = np.array([8.2, 7.9, 7.5, 7.4])
print(celsius * 9 / 5 + 32)
print(celsius.mean())
```

```
[46.76 46.22 45.5  45.32]
7.75
```

One multiplication, one division and one addition were written, and four readings were converted.
`celsius.mean()` added the four and divided by four. Neither line has a loop in it, and neither would
get any longer for 35,040 readings.

## Setup

Four imports, and a year of readings as Python lists.

- `numpy`, imported as `np` as nearly every program that uses it does, is the library, and the cell
  prints its version
- `math`, `datetime` and `timedelta` build the readings, and `START + timedelta(hours=n)` turns an
  hour's position in the year back into a date and time
- `timeit` times the loops against the array, in the last worked example

`readings_list` holds four lists, one for each station in `STATIONS`, with a reading for every hour of
2025. `year_of_readings` makes them by the same formula as the **sqlite3, Deep Dive** guide, from each
station's mean, a seasonal swing, a daily swing and a small wobble. Svalbard's 24 readings for
2 March are `None`, because that station sent nothing that day.

In [1]:
import math
import timeit
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]

print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each")

numpy 2.1.3 | 4 stations, 8760 hours each


## Worked examples

### The readings as Python lists

`readings_list` is a list of four lists. Here are Bergen's first six hours, the length of every list,
and the four Svalbard readings around midnight on 2 March, when the station went silent:

In [2]:
print(readings_list[0][:6])
print([len(row) for row in readings_list])
print(readings_list[2][1438:1442])                    # 22:00 and 23:00 on 1 March, then 00:00 and 01:00

[-3.6, -3.7, -3.7, -3.5, -3.1, -2.5]
[8760, 8760, 8760, 8760]
[-11.6, -12.0, None, None]


### Every reading in Fahrenheit with loops

Converting a list means building a new one, a value at a time. The `None` hours have to be passed
through untouched, because `None * 9` raises an error:

In [3]:
fahrenheit_list = []
for row in readings_list:
    converted = []
    for celsius in row:
        if celsius is None:
            converted.append(None)
        else:
            converted.append(celsius * 9 / 5 + 32)
    fahrenheit_list.append(converted)

print(fahrenheit_list[0][:6])

[25.52, 25.34, 25.34, 25.7, 26.42, 27.5]


Two loops and an `if` walk through the data, and the formula is one line in the middle of them.

### The same readings as one array

`np.array` turns the list of lists into an array. `dtype=float` asks for decimals, which turns every
`None` into `nan`:

In [4]:
readings = np.array(readings_list, dtype=float)

print(readings.shape)
print(readings.dtype)
print(readings[0, :6])
print(readings[2, 1438:1442])

(4, 8760)
float64
[-3.6 -3.7 -3.7 -3.5 -3.1 -2.5]
[-11.6 -12.    nan   nan]


`shape` is (4, 8760), four stations by 8,760 hours, and `dtype` is `float64`, a decimal stored in
64 bits. `readings[0, :6]` reads row 0, the first six columns. The **Indexing and Slicing** notebook
covers that notation, and the **dtype** notebook what else an array can hold.

### Every reading in Fahrenheit at once

In [5]:
fahrenheit = readings * 9 / 5 + 32

print(fahrenheit.shape)
print(fahrenheit[0, :6])
print(fahrenheit[2, 1438:1442])

(4, 8760)
[25.52 25.34 25.34 25.7  26.42 27.5 ]
[11.12 10.4    nan   nan]


The formula is the whole program. `nan` went in and `nan` came out, so the missing hours needed no
`if`. The loops and the array hold the same numbers, which `np.array_equal` confirms, with
`equal_nan=True` so that two `nan` values count as equal:

In [6]:
print(np.array_equal(np.array(fahrenheit_list, dtype=float), fahrenheit, equal_nan=True))

True


### Each station's mean

The loop version adds up the readings that are there and divides by how many there are:

In [7]:
for name, row in zip(STATIONS, readings_list):
    present = [celsius for celsius in row if celsius is not None]
    print(name, round(sum(present) / len(present), 2))

Bergen 8.0
Oslo 6.5
Svalbard -4.48
Tromso 3.5


An array has a `.mean()` method, and NumPy has `np.nanmean`, which leaves `nan` out:

In [8]:
for name, row in zip(STATIONS, readings):
    print(name, round(row.mean(), 2), round(np.nanmean(row), 2))

Bergen 8.0 8.0
Oslo 6.5 6.5
Svalbard nan -4.48
Tromso 3.5 3.5


Svalbard's `.mean()` is `nan`, because one `nan` in a sum makes the whole sum `nan`. `np.nanmean`
leaves the missing hours out of both the total and the count, which is what the loop's `if` did. The
**Aggregation** notebook covers the rest of the functions that skip `nan`.

### The coldest hour

The loop has to remember two things as it goes, the lowest reading so far and its position:

In [9]:
row = readings_list[2]                                # Svalbard
coldest, where = None, None
for n, celsius in enumerate(row):
    if celsius is not None and (coldest is None or celsius < coldest):
        coldest, where = celsius, n

print(coldest, START + timedelta(hours=where))

-17.3 2025-01-12 03:00:00


`np.nanargmin` returns the position of the lowest value, skipping `nan`:

In [10]:
where = np.nanargmin(readings[2])

print(readings[2, where], START + timedelta(hours=int(where)))

-17.3 2025-01-12 03:00:00


A position along a station's row is an hour of the year, so `START + timedelta(hours=...)` turns it
into a date and time. `int(where)` turns NumPy's integer into Python's, because `timedelta` takes
only Python's.

### Daily means in one line

With lists, a mean for every day of every station is a third loop. With the array it is one line:

In [11]:
daily = readings.reshape(4, 365, 24).mean(axis=2)

print(daily.shape)
print(daily[0, :3].round(2))
print("days that are nan:", np.isnan(daily).sum(), "| Svalbard on 2 March:", daily[2, 60])

(4, 365)
[-0.7  -0.7  -0.73]
days that are nan: 1 | Svalbard on 2 March: nan


`reshape(4, 365, 24)` regroups each station's 8,760 hours as 365 days of 24, and `.mean(axis=2)`
averages along the last of those three dimensions, the hours. One daily mean is `nan`, Svalbard's
for 2 March, the 61st day, whose hours are all missing. The **Shape** notebook takes `reshape` apart,
and **The axis Argument** notebook the `axis`.

### Timing the loops against the array

Speed is the other reason to use an array. `timeit.repeat` runs each version three times in a row,
five times over, and `min` keeps the fastest of the five. The seconds depend on the machine, so the
cell prints only whether the loops took more than five times as long as the array:

In [12]:
def with_loops():
    return [[None if celsius is None else celsius * 9 / 5 + 32 for celsius in row] for row in readings_list]


def with_an_array():
    return readings * 9 / 5 + 32


loop_time = min(timeit.repeat(with_loops, number=3, repeat=5))
array_time = min(timeit.repeat(with_an_array, number=3, repeat=5))
print("the loops took more than five times as long:", loop_time > 5 * array_time)

the loops took more than five times as long: True


`with_loops` is the shortest loop version there is, a list comprehension, and it still lost. The
**Performance** notebook measures properly, and shows where the difference comes from.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/01-why-numpy-solutions.ipynb).

**1.** Make an array from the list `[12.5, 13.0, 11.75]`, and print it, its shape and its dtype.

In [13]:
# your code here

**2.** Convert Oslo's readings, row 1 of `readings`, to kelvin by adding 273.15, and print the first five.

In [14]:
# your code here

**3.** Find Tromso's warmest hour of the year, row 3, and print the reading with its date and time.

In [15]:
# your code here

**4.** Print every station's lowest reading of the year, leaving Svalbard's missing hours out.

In [16]:
# your code here

**5.** Make an array of the difference between Bergen and Oslo at every hour, and print its mean rounded to two places.

In [17]:
# your code here

## Common errors

### TypeError: can't multiply sequence by non-int of type 'float'

In [18]:
readings_list[0] * 1.8

TypeError: can't multiply sequence by non-int of type 'float'

A list does no arithmetic. `*` with a list means repetition, and you can repeat a list only a whole
number of times, so a decimal raises. `np.array(readings_list[0]) * 1.8` makes an array first, and
multiplies every reading by 1.8.

### No error, and a list twice as long

In [19]:
doubled = readings_list[0] * 2

print(len(readings_list[0]), len(doubled))

8760 17520


With a whole number, `*` repeats the list. `doubled` holds Bergen's year twice over, one copy after
the other, and nothing warns you. An array times 2 doubles every value instead.

### TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'

In [20]:
raw = np.array(readings_list)
print(raw.dtype)

raw * 9 / 5 + 32

object


TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'

Without `dtype=float`, NumPy found `None` among the numbers and made an array of Python objects,
`dtype` `object`. Arithmetic on such an array runs Python's own operations one value at a time, and
the first `None` raised. `np.array(readings_list, dtype=float)` turns every `None` into `nan`, and
arithmetic passes `nan` through.

### No error, and a mean of nan

In [21]:
print(readings[2].mean())
print(np.nanmean(readings[2]))

nan
-4.481902472527473


`.mean()` adds every value, and a single `nan` makes the total `nan`. Svalbard has 24 of them. Use
`np.nanmean`, or the other `nan` functions, wherever readings can be missing.

### ValueError: setting an array element with a sequence.

In [22]:
short_year = [readings_list[0], readings_list[1][:-1]]     # Oslo's list one hour short

np.array(short_year, dtype=float)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.

Every row of an array is the same length. Oslo's list here is one hour short, so the two lists cannot
make a grid, and the rest of the message says that the shapes stopped agreeing after the first
dimension, the one that counts the two stations. Printing `[len(row) for row in short_year]` shows which list is the short one.

## Recap

- Make an array from a list of lists with `np.array(..., dtype=float)`, and read its `shape` and
  `dtype`
- Apply a formula to every value at once, with no loop and no `if` for the missing values
- Find a mean, a lowest value and its position with `np.nanmean`, `np.nanmin` and `np.nanargmin`
- Use `nan` as NumPy's missing value, and choose the functions that skip it
- Tell what `*` does to a list from what it does to an array

## What is next

The **Creating Arrays** notebook makes arrays without a list to start from: zeros, ranges and evenly
spaced values, and the type each one chooses.

---

[NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
